# Coding Attention Mechanisms

In [2]:
from importlib.metadata import version

from sympy import false

print("torch version:", version("torch"))

torch version: 2.12.0


- The code below walks through the figure above step by step

<br>

- **Step 1:** compute unnormalized attention scores $\omega$
- Suppose we use the second input token as the query, that is, $q^{(2)} = x^{(2)}$, we compute the unnormalized attention scores via dot products:
    - $\omega_{21} = x^{(1)} q^{(2)\top}$
    - $\omega_{22} = x^{(2)} q^{(2)\top}$
    - $\omega_{23} = x^{(3)} q^{(2)\top}$
    - ...
    - $\omega_{2T} = x^{(T)} q^{(2)\top}$
- Above, $\omega$ is the Greek letter "omega" used to symbolize the unnormalized attention scores
    - The subscript "21" in $\omega_{21}$ means that input sequence element 2 was used as a query against input sequence element 1

In [3]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [8]:
query = inputs[1] #A
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [9]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


In [10]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [11]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [12]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [25]:
query = inputs[1] # 2nd input token is the query
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    # print(context_vec_2)
    # print(attn_weights_2[i]*x_i)
    # 1. 权重 × 对应词向量：加权向量
    # 2. 累加到上下文向量（迭代求和）
    context_vec_2 += attn_weights_2[i]*x_i
    # print(context_vec_2)
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [14]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [16]:
attn_scores = inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [18]:
attn_weights = torch.softmax(attn_scores, dim=1)
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [19]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)
print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [20]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


首先定义几个我们要使用到的变量

In [27]:
x_2 =  inputs[1]
d_in = inputs.shape[1]
d_out = 2
print(x_2)
print(d_in)

tensor([0.5500, 0.8700, 0.6600])
3


requires_grad 设置为 False，以便在输出结果中减少不必要的信息，从而使演示更加清晰。但如果要将这些权重矩阵用于模型训练，则需要将 requires_grad 设置为 True，以便在模型训练过程中更新这些矩阵

In [29]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)


In [30]:
W_query

Parameter containing:
tensor([[-0.1115,  0.1204],
        [-0.3696, -0.2404],
        [-1.1969,  0.2093]])

In [31]:
W_key

Parameter containing:
tensor([[-0.9724, -0.7550],
        [ 0.3239, -0.1085],
        [ 0.2103, -0.3908]])

In [32]:
W_value

Parameter containing:
tensor([[ 0.2350,  0.6653],
        [ 0.3528,  0.9728],
        [-0.0386, -0.8861]])

In [33]:
query_2 = x_2 @ W_query
query_2

tensor([-1.1729, -0.0048])

In [34]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)
print(key_2)
print(value_2)

tensor([-1.1729, -0.0048])
tensor([-0.1142, -0.7676])
tensor([0.4107, 0.6274])


In [40]:
keys = inputs @ W_key
value = inputs @ W_value

print(keys.shape)
print(keys.shape[-1])
print(keys)

torch.Size([6, 2])
2
tensor([[-0.1823, -0.6888],
        [-0.1142, -0.7676],
        [-0.1443, -0.7728],
        [ 0.0434, -0.3580],
        [-0.6467, -0.6476],
        [ 0.3262, -0.3395]])


In [36]:
print(value)

tensor([[ 0.1196, -0.3566],
        [ 0.4107,  0.6274],
        [ 0.4091,  0.6390],
        [ 0.2436,  0.4182],
        [ 0.2653,  0.6668],
        [ 0.2728,  0.3242]])


#### 计算注意力分数

In [38]:
keys_2 = keys[1]
print(query_2)
print(keys_2)
attn_scores_22 =  torch.dot(query_2, keys_2)
attn_scores_22

tensor([-1.1729, -0.0048])
tensor([-0.1142, -0.7676])


tensor(0.1376)

In [42]:
attn_scores_2 = query_2 @ keys.T
attn_scores_2

tensor([ 0.2172,  0.1376,  0.1730, -0.0491,  0.7616, -0.3809],
       grad_fn=<SqueezeBackward4>)

#### 计算标准化的注意力权重

In [58]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
attn_weights_2

tensor([0.1704, 0.1611, 0.1652, 0.1412, 0.2505, 0.1117],
       grad_fn=<SoftmaxBackward0>)

In [44]:
attn_weights_2.sum()

tensor(1.0000, grad_fn=<SumBackward0>)

#### 计算上下文向量

In [39]:
print(attn_weights_2)
print(value)
context_vec_2 = attn_weights_2 @ value
context_vec_2

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor([[ 0.1196, -0.3566],
        [ 0.4107,  0.6274],
        [ 0.4091,  0.6390],
        [ 0.2436,  0.4182],
        [ 0.2653,  0.6668],
        [ 0.2728,  0.3242]])


tensor([0.3117, 0.4242])

In [41]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
        self.W_key = nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
        self.W_value = nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)


    def forward(self, x):
        # 计算 K 向量
        keys = x @ self.W_key
        # 计算 Q 向量
        queries = x @ self.W_query
        # 计算 value 向量
        values = x @ self.W_value

        # 计算注意力分数
        attn_scores = queries @ keys.T
        # 转成 注意力权重
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        # 计算上线文向量
        context_vec = attn_weights @ values
        # 返回上下文向量
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2845, 0.4071],
        [0.2854, 0.4081],
        [0.2854, 0.4075],
        [0.2864, 0.3974],
        [0.2863, 0.3910],
        [0.2860, 0.4039]])


In [42]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))


tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


### 实现因果自主注意力机制

In [45]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
values = sa_v2.W_value(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [46]:
context_length = attn_scores.shape[0]
# 生成一个下三角矩阵（包含对角线），未来位置为 0 过去位置为 1
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [47]:
# 把未来额度注意力权重变为 0
mask_simple = attn_weights * mask_simple
print(mask_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [48]:
# 重新归一化
row_sums = mask_simple.sum(dim=1, keepdim=True)
print(row_sums)
# 使每行的权重加起来等于 1
masked_simple_norm = mask_simple / row_sums
print(masked_simple_norm)

tensor([[0.1921],
        [0.3700],
        [0.5357],
        [0.6775],
        [0.8415],
        [1.0000]], grad_fn=<SumBackward1>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [50]:
# 第二种实现：符合 Transformer 论文的官方实现
# 生成一个上三角矩阵（不含对角线），未来位置为 1，过去位置为 0
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
# 在计算 Softmax 之前，把所有未来位置的分数变成 -inf
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [52]:
# Softmax 自动将这些位置的权重归零，且不需要手动重新归一化，因为剩余数的指数和自然等于 1。
print(keys.shape[-1])
print(keys.shape[-1]**0.5)
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

2
1.4142135623730951
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [55]:
torch.manual_seed(123)
# 我们使用的dropout率为0.5
dropout = torch.nn.Dropout(0.5)
# 创建一个由1组成的矩阵
example = torch.ones(6, 6)
print(example)
print(dropout(example))


tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])
tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [78]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


In [76]:
# Listing 3.3 A compact causal attention class
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        # print("d_in", d_in)  # 3
        # print("d_out", d_out) # 2
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)  # 生成的权重矩阵是 2 行 3 列
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # 与之前的 SelfAttention_v1 类相比，我们添加了一个 dropout 层
        self.dropout = nn.Dropout(dropout)                        #A
        # register_buffer 调用也是新添加的内容（后续内容会提供更多相关信息）
        # egister_buffer 会告诉 PyTorch
        # 这个张量不是模型参数（不需要梯度），但必须随模型一起保存（state_dict）并自动跟随模型移动到 GPU 或 CPU。
        self.register_buffer(
           'mask',
           torch.triu(torch.ones(context_length, context_length),
           diagonal=1)
        )                                                         #B

    def forward(self, x):
        '''
         x 的形状为 (b, num_tokens, d_in)（批次、序列长度、嵌入维度）
        '''
        # 我们交换第 1 和第 2 个维度，同时保持批次维度在第1个位置（索引0）
        b, num_tokens, d_in = x.shape                             #C
        print("---- x shape", x.shape)
        # 拿出 W_key 权重，形状 (2, 3)，转置为 (3, 2)。
        # 批量相乘：(2, 6, 3) @ (3, 2)，消掉中间的 3。
        # 拼形状：把剩下的维度拼起来，就是 (2, 6, 2)。
        # 广播加偏置：如果 bias=True，再加个形状 (2,) 的数。
        # 这里 模拟的是 wx + b
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        # queries @ keys.transpose(1, 2) -> (b, num_tokens, num_tokens)
        # keys 形状 (2, 4, 512) -> transpose(1,2) 后变成 (2, 512, 4)
        attn_scores = queries @ keys.transpose(1, 2)              #C
        # 在 PyTorch 中，带有下划线后缀的操作会在原有内存空间执行，直接修改变量本身，从而避免不必要的内存拷贝
        # elf.mask.bool()[:num_tokens, :num_tokens]：从预先存好的 context_length x context_length 大掩码中，
        # 裁切出当前输入实际长度的子矩阵。这允许同一个模型处理不同长度的句子（只要不超过预设的 context_length）
        # 将上三角位置（未来词）填充为 -torch.inf
        attn_scores.masked_fill_(                                 #D
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        # 缩放
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        # 随机将部分注意力权重置零（注意：权重已经是概率分布，置零后会自动缩放）
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

In [77]:
# 将两个完全相同的 inputs 张量沿新的第 0 维堆叠起来。
# 假设你之前定义的 inputs 是形状为 (n, d_in) 的二维矩阵（n 个词，每个词 d_in 维）。
# 堆叠后，batch 变成形状为 (2, n, d_in) 的三维张量
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)
torch.manual_seed(123)
# 序列的长度
context_length = batch.shape[1]
# 这里将 Dropout 概率设为 0.0。这意味着 Dropout 层完全失效（相当于直接返回输入），没有任何随机性
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

torch.Size([2, 6, 3])
---- x shape torch.Size([2, 6, 3])
context_vecs.shape: torch.Size([2, 6, 2])


### 多头自注意力机制的实现

In [78]:
class MultiHeadAttentionWrapper(nn.Module):

    '''
    d_in：输入嵌入维度（每个 token 的特征数）。
    d_out：每个头的输出维度（即每个头内部 Linear 层的输出大小）。
    context_length：最大序列长度（用于预生成因果掩码矩阵）。
    dropout：注意力权重的 dropout 率。
    num_heads：注意力头的数量。
    qkv_bias：是否在 Q/K/V 线性层中使用偏置
    '''
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        # 将列表中的张量，在 dim=-1（即最后一个维度） 上进行拼接。
        # 目的: 将多个独立头在子空间中挖掘到的异构特征无损地汇总成一个高维向量
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

---- x shape torch.Size([2, 6, 3])
---- x shape torch.Size([2, 6, 3])
tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


In [79]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # 输出维度要能被头数整除
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"
        # d_out 是所有头合并后的总输出维度，而不是每个头的输出维度。每个头实际工作的维度是 head_dim
        self.d_out = d_out
        self.num_heads = num_heads
        # 上下文向量的维度
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # 输出层投影
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            # torch.ones 生成一个全是 1 的方阵
            # torch.triu(..., diagonal=1) 保留上三角部分（不含对角线），其余位置置 0
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )


    def forward(self, x):
        '''
        view 拆分：(b, seq, d_out) -> (b, seq, heads, dim)（逻辑分组，未移动内存）。
        transpose(1, 2)：(b, seq, heads, dim) -> (b, heads, seq, dim)（把“头”提到前面，便于批量并行）。
        transpose(2, 3)（仅对 keys）：(b, heads, seq, dim) -> (b, heads, dim, seq)（调换特征维度和序列维度，只为对齐矩阵乘法的中间轴）。

        attn_scores      : (b, heads, seq, seq)
        ↓ 缩放 + softmax
        attn_weights     : (b, heads, seq, seq)
        ↓ dropout (训练)
        attn_weights     : (b, heads, seq, seq)
        ↓ @ values (加权求和)
        temp             : (b, heads, seq, head_dim)
        ↓ transpose(1,2)
        temp             : (b, seq, heads, head_dim)
        ↓ contiguous + view
        context_vec      : (b, seq, d_out)
        ↓ out_proj (融合)
        context_vec      : (b, seq, d_out)   # 最终输出
        '''
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`,
        # this will result in errors in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forwar

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        # 执行 view 拆分后，我们虽然在逻辑上区分了头，但此时维度顺序是 (批次, 序列, 头, 维度)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        # transpose(1, 2)，将(批次, 序列, 头, 维度) -> (批次, 头, 序列, 维度)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        # 互不干扰地计算出所有头的注意力分数  keys.transpose(2, 3) 将(批次, 头, 序列, 维度) -> (批次, 头, 维度, 序列)
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        # .bool()：将之前的 0 和 1 转换为布尔值 False 和 True（1 -> True，0 -> False）。
        # [:num_tokens, :num_tokens]：裁剪出当前输入序列实际长度对应的子矩阵。
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        # attn_scores 的形状是 (b, num_heads, num_tokens, num_tokens)
        # masked_fill_（带下划线表示原地操作）会遍历 attn_scores。
        # 它会在掩码为 True 的位置（即未来位置），将原本的注意力分数替换成 -torch.inf（负无穷大）。
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # 计算注意力权重
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        # 随机遮蔽掉一些信息
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        # (b, num_heads, num_tokens, num_tokens) @ (b, num_heads, num_tokens, head_dim)
        # → (b, num_heads, num_tokens, head_dim)
        # 对于每个头、每个批次、每个查询位置，将 attn_weights 作为权重，对所有的 values 进行加权平均。
        # 结果是每个位置都获得了一个融合了全局信息的“上下文向量”。
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        # .contiguous()：由于上一步的 .transpose 改变了张量的内存排列（产生了“非连续视图”），
        # 后续的 .view 操作要求内存必须是连续的。contiguous() 强制在物理内存上重新排列数据，使其变得连续
        # .view(b, num_tokens, self.d_out)：将最后两维 (num_heads, head_dim) “拉平”为 num_heads * head_dim = self.d_out。
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        # 至此，所有头的输出被物理拼接在一起，形状变回 (b, num_tokens, d_out)，回到了标准的特征维度。
        # self.out_proj 是一个线性层，输入输出维度均为 d_out
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


In [81]:
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],             #A
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],
                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

#A 该张量的形状为 (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
print(a.shape)

torch.Size([1, 2, 3, 4])


In [85]:
# print(a.transpose(2,3))
print(a @ a.transpose(2, 3))


tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In [86]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)
second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


In [88]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
print(batch.shape)
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)


torch.Size([2, 6, 3])
tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
